# 🎯 [DAT10_M04] 네트워크 복구 시 클라우드 중앙 서버와의 양방향 데이터 충돌 해결(CRDT) 동기화 고도화
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT10/dat10_m04_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 3) |
| **미션 ID** | DAT10_M04 |
| **미션 주제** | 네트워크 복구 시 클라우드 중앙 서버와의 양방향 데이터 충돌 해결(CRDT) 동기화 고도화 |
| **소요 시간** | 약 3.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
전시에는 전술 통신망이 단절된 채 예하 보급소가 입고와 불출을 각자 기록하고, 통신 복구 후 기록을 병합하는 과정에서 중복·누락·충돌·위변조가 빈번히 발생하며, 이를 적시에 통제하지 못하면 재고 왜곡과 방산 공급망 위기 식별이 지연됩니다. 본 프로젝트는 방산 공급망 회복력 연구(미국 방위산업기반 정책연구)의 문제의식을 교육용 가상 자료로 재구성해, 가상 보급소 5곳이 통신 두절 환경에서 오프라인으로 재고를 기록하고 동기화하는 SaaS를 개발하고, 동기화된 데이터에서 공급망 재난 징후(업체 공급 감소, 재고 소진 임박)를 식별하는 End-to-End 군수 소프트웨어 엔지니어링 과제입니다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 04. 네트워크 복구 시 클라우드 중앙 서버와의 양방향 데이터 충돌 해결(CRDT) 동기화 고도화
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

네트워크 복구 시 중앙 서버와의 양방향 데이터 충돌 해결(CRDT) 동기화를 고도화한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **통신 복구 후 노드와 중앙 서버가 양방향으로 병합하는 CRDT 동기화를 구현하고, 같은 품목 정보를 서로 모르게 고친 충돌의 탐지 재현율 85% 이상과 처리 지연 200ms 이하를 확인한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **재고 수량은 품목별 PN-카운터(노드별 증가·감소 누적, 병합은 요소별 최댓값)로 만들어 노드 5곳과 서버를 2단계로 병합하고, 마지막 값이 이기는 방식(LWW)과 갱신 유실을 비교 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/models/dat10_advanced_detector.py 및 예외 탐지 테스트 결과` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 기만/교란 공격에 대한 방어 로직의 현실성 및 신뢰성

#### 미션 PASS 조건

- 정량 기준: 복합 위협 탐지 재현율(Recall) 85% 이상, 처리 지연 200ms 이하


- 진행 기준: 복합 위협 시나리오 10종에 대한 탐지 테스트 통과


- 검증 방법: 합성 위협 데이터셋 주입 시 이상치 탐지 정확도 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""통신 복구 후 보급소 5곳과 중앙 서버가 양방향으로 병합하는 CRDT 동기화(재고는 PN-카운터, 메타 정보는 버전 벡터 기반 충돌 감지)를 구현해, 수렴·갱신 유실 0건·멱등/교환/결합 성질·중복 재전송·오래된 상태 재전송 방어를 검증하고, 충돌 탐지 재현율(시간 비교 규칙 대 버전 벡터 규칙)과 오경보 패턴·동기화 지연을 측정한다."""
import copy
import itertools
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 야전 보급소·업체·품목 자료 생성기 (실제 부대·업체·품목·재고 정보 없음)
# 가상 보급소 5곳이 품목 12종의 입고·불출·조정 트랜잭션을 오프라인으로 기록하고 나중에 동기화한다.
# 공급망 재난 징후(업체 납기 지연, 품목 재고 급감)는 동기화된 자료를 마이닝해 식별한다.
# ---------------------------------------------------------------------------
NODES = ["보급소A", "보급소B", "보급소C", "보급소D", "보급소E"]
SUPPLIERS = ["가상업체1", "가상업체2", "가상업체3", "가상업체4", "가상업체5", "가상업체6"]
CATS = ["일반보급", "정비부품", "의무소모"]
ITEMS = [{"item": f"I{i + 1:02d}", "cat": CATS[i % 3], "supplier": SUPPLIERS[i % 6], "lead_days": 3 + (i * 5) % 12, "use": 4 + (i * 7) % 11, "min_level": 30 + 10 * (i % 5)} for i in range(12)]
ITEM_IDS = [x["item"] for x in ITEMS]
CTXS = ["평시", "야간", "통신불량"]


def make_txns(seed=7, days=7, per_node_day=60, nodes=None):
    """노드별 오프라인 트랜잭션 목록. id는 노드-일련번호(멱등 키), t는 시각(시간 단위), ctx는 입력 상황."""
    rng = np.random.default_rng(seed)
    out = []
    for n in (nodes or NODES):
        seq = 0
        for k in range(days * per_node_day):
            seq += 1
            it = ITEMS[int(rng.integers(len(ITEMS)))]
            t = float(rng.uniform(0, 24 * days))
            ctx = "야간" if (t % 24 >= 20 or t % 24 < 5) else "평시"
            if rng.random() < 0.12:
                ctx = "통신불량"
            r = rng.random()
            if r < 0.22:
                op, q = "in", int(it["use"] * rng.uniform(4, 9))
            elif r < 0.95:
                op, q = "out", int(max(1, rng.normal(it["use"] * 1.5, 3)))
            else:
                op, q = "adjust", int(rng.integers(-5, 6))
            out.append({"id": f"{n}-{seq:05d}", "node": n, "item": it["item"], "op": op, "qty": q, "t": round(t, 2), "ctx": ctx, "supplier": it["supplier"]})
    return out


def stock_of(txns, base=500):
    """품목별 재고 = 기초 재고 + 입고 - 불출 + 조정 (id 중복은 한 번만 반영)."""
    # [TODO : 학생 실습 4] 네트워크 복구 시 클라우드 중앙 서버와의 양방향 데이터 충돌 해결(CRDT) 동기화 고도화 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
class Replica:
    """재고는 품목별 PN-카운터(노드별 증가량 P, 감소량 N), 같은 노드의 기록은 서로 덮어쓰지 않는다. 병합은 요소별 최댓값."""
    def __init__(self, name):
        self.name = name; self.P = {i: {} for i in ITEM_IDS}; self.N = {i: {} for i in ITEM_IDS}

    def apply(self, x):
        d = self.P if x["op"] == "in" or (x["op"] == "adjust" and x["qty"] > 0) else self.N
        d[x["item"]][x["node"]] = d[x["item"]].get(x["node"], 0) + abs(x["qty"])

    def merge(self, other):
        for it in ITEM_IDS:
            for src, dst in ((other.P, self.P), (other.N, self.N)):
                for n, v in src[it].items():
                    dst[it][n] = max(dst[it].get(n, 0), v)

    def stock(self, base=500):
        return {i: base + sum(self.P[i].values()) - sum(self.N[i].values()) for i in ITEM_IDS}

    def state(self):
        return (tuple(tuple(sorted(self.P[i].items())) for i in ITEM_IDS), tuple(tuple(sorted(self.N[i].items())) for i in ITEM_IDS))


TX = make_txns(7, 7, per_node_day=60)
truth = stock_of(TX)
reps = {n: Replica(n) for n in NODES}
for x in TX:
    reps[x["node"]].apply(x)                                       # 7일간 각 노드는 자기 기록만 반영(오프라인)
server = Replica("서버")

# 1) 통신 복구: 노드 -> 서버 병합 후 서버 -> 노드 병합(2단계). 지연 측정
lat = []
t0 = time.perf_counter()
for n in NODES:
    t1 = time.perf_counter(); server.merge(reps[n]); lat.append((time.perf_counter() - t1) * 1000)
for n in NODES:
    t1 = time.perf_counter(); reps[n].merge(server); lat.append((time.perf_counter() - t1) * 1000)
dt = time.perf_counter() - t0
conv = len({r.state() for r in reps.values()} | {server.state()}) == 1
lost = sum(abs(server.stock()[i] - truth[i]) for i in ITEM_IDS)
print(f"[CRDT 동기화] 노드 5곳 + 서버, 트랜잭션 {len(TX)}건: 수렴 {'성공' if conv else '실패'}, 서버 재고와 정답(전체 합산) 차이 합계 {lost}개 -> 갱신 유실 {'0건' if lost == 0 else '있음'}")
print(f"병합 1회 평균 {np.mean(lat):.3f}ms, 최대 {max(lat):.3f}ms, 전체 동기화 {dt * 1000:.2f}ms (목표 200ms 이하: {'달성' if max(lat) <= 200 else '미달'})")

# 2) 비교: 재고 숫자를 '마지막 기록이 이김(LWW)'으로 덮어쓰면 오프라인 갱신이 사라진다
last = {}
for n in NODES:                                                    # 각 노드의 최종 재고 값(기초 재고 + 자기 기록만 반영)
    loc = {i: 500 for i in ITEM_IDS}
    for x in TX:
        if x["node"] == n:
            loc[x["item"]] += x["qty"] if x["op"] in ("in", "adjust") else -x["qty"]
    last[n] = (max(x["t"] for x in TX if x["node"] == n), loc)
lww = max(last.values())[1]
print(f"숫자 덮어쓰기(LWW) 병합: 정답과 차이 합계 {sum(abs(lww[i] - truth[i]) for i in ITEM_IDS)}개 (갱신 유실) / PN-카운터 {lost}개")

# 3) CRDT 성질 시험: 멱등·교환·결합·중복·오래된 상태 재전송
rng = np.random.default_rng(5); ok = {"멱등(두 번 병합)": 0, "교환(A+B = B+A)": 0, "결합((A+B)+C = A+(B+C))": 0, "오래된 상태 재전송": 0}
trials = 200
for _ in range(trials):
    a, b, c = (reps[n] for n in rng.choice(NODES, 3, replace=False))
    parts = []
    for r in (a, b, c):
        q = Replica("t")
        for i in ITEM_IDS:
            for n in NODES:
                if rng.random() < .5:
                    q.P[i][n] = int(rng.integers(0, 100)); q.N[i][n] = int(rng.integers(0, 100))
        parts.append(q)
    x, y, z = parts
    m = copy.deepcopy(x); m.merge(y); s1 = m.state(); m.merge(y); ok["멱등(두 번 병합)"] += s1 == m.state()
    m1 = copy.deepcopy(x); m1.merge(y); m2 = copy.deepcopy(y); m2.merge(x); ok["교환(A+B = B+A)"] += m1.state() == m2.state()
    l = copy.deepcopy(x); l.merge(y); l.merge(z); rr = copy.deepcopy(y); rr.merge(z); r2 = copy.deepcopy(x); r2.merge(rr); ok["결합((A+B)+C = A+(B+C))"] += l.state() == r2.state()
    new = copy.deepcopy(x); new.merge(y); old = copy.deepcopy(x); before = new.state(); new.merge(old); ok["오래된 상태 재전송"] += before == new.state()
print("\n[CRDT 성질 시험(무작위 상태 200쌍)]", ", ".join(f"{k} {v}/{trials}" for k, v in ok.items()))

# 4) 메타 정보(최소 보유량·위치·상태) 동시 수정 충돌 탐지: 시간 비교 규칙 vs 버전 벡터 규칙
def dominated(a, b):
    # [TODO : 학생 실습 4] 네트워크 복구 시 클라우드 중앙 서버와의 양방향 데이터 충돌 해결(CRDT) 동기화 고도화 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_pair(kind):
    base = float(rng.uniform(0, 160)); va, vb = int(rng.integers(30, 80)), int(rng.integers(30, 80))
    if va == vb:
        vb += 7
    A = {"field": "min_level", "value": va, "t": base, "vv": {"A": 3, "B": 1}}
    if kind == "concurrent":
        B = {"field": "min_level", "value": vb, "t": base + float(rng.uniform(0.5, 120)), "vv": {"A": 2, "B": 4}}
    elif kind == "causal":                                      # B가 A의 수정을 받은 뒤 고침(정상 후속 수정), 시각은 가까움
        B = {"field": "min_level", "value": vb, "t": base + float(rng.uniform(0.01, 0.5)), "vv": {"A": 3, "B": 2}}
    elif kind == "same_value":
        B = {"field": "min_level", "value": va, "t": base + float(rng.uniform(0.5, 120)), "vv": {"A": 2, "B": 4}}
    elif kind == "other_field":
        B = {"field": "location", "value": vb, "t": base + float(rng.uniform(0.5, 120)), "vv": {"A": 2, "B": 4}}
    else:                                                        # 시계 어긋남: 실제로는 동시 수정이지만 B 단말 시계가 느려 시각이 A보다 앞선 것처럼 기록됨
        B = {"field": "min_level", "value": vb, "t": base - float(rng.uniform(0.5, 3)), "vv": {"A": 2, "B": 4}}
    return A, B


truth_conf = {"concurrent": True, "causal": False, "same_value": False, "other_field": False, "skew": True}
pairs = [(k, *make_pair(k)) for k in rng.choice(list(truth_conf), 400, p=[.40, .25, .10, .10, .15])]
rule_time = lambda A, B: A["field"] == B["field"] and A["value"] != B["value"] and abs(A["t"] - B["t"]) <= 1.0          # 1시간 이내 같은 필드 다른 값
rule_val = lambda A, B: A["field"] == B["field"] and A["value"] != B["value"]                                  # 같은 필드 다른 값이면 모두 충돌(인과 무시)
rule_vv1 = lambda A, B: A["field"] == B["field"] and not dominated(A["vv"], B["vv"]) and not dominated(B["vv"], A["vv"])      # 동시이면 충돌(값 비교 없음)
rule_vv2 = lambda A, B: rule_vv1(A, B) and A["value"] != B["value"]                                                               # 재설계: 값이 다를 때만
print("\n[충돌 탐지] 정답 충돌 %d건 / 쌍 %d건" % (sum(truth_conf[k] for k, _, _ in pairs), len(pairs)))
for name, rule in (("시간 근접 규칙(1시간 이내)", rule_time), ("값 비교 규칙(다르면 충돌)", rule_val), ("버전 벡터 규칙 v1(동시이면 충돌)", rule_vv1), ("버전 벡터 + 값 비교 v2(재설계)", rule_vv2)):
    pred = np.array([rule(a, b) for _, a, b in pairs]); tr = np.array([truth_conf[k] for k, _, _ in pairs])
    tp, fp, fn = int((pred & tr).sum()), int((pred & ~tr).sum()), int((~pred & tr).sum())
    kinds = {str(k): int(v) for k, v in pd.Series([k for (k, _, _), p, t in zip(pairs, pred, tr) if p != t]).value_counts().items()}
    print(f"  {name:30s} 재현율 {tp / (tp + fn):5.1%}, 정밀도 {tp / max(tp + fp, 1):5.1%}, 오경보 {fp}건, 놓침 {fn}건 {kinds}")
print("오경보·미탐 패턴: 시간 규칙은 오프라인 동안 멀리 떨어진 동시 수정(concurrent)·시계 어긋남(skew)을 놓치고, 값 비교 규칙은 '후속 정상 수정'(causal)을 충돌로 올림 -> 버전 벡터로 인과 관계를 보고 같은 값이면 제외(v2)")
print("※ 충돌로 표시된 건은 자동 병합하지 않고 군수관이 수동 병합한다(M7). 재고 수량은 CRDT로 자동 합산되므로 사람이 판단할 대상은 메타 정보 충돌이다.")

plt.figure(figsize=(6, 3.5)); x = np.arange(len(ITEM_IDS))
plt.bar(x - .2, [truth[i] for i in ITEM_IDS], .4, label="truth / CRDT"); plt.bar(x + .2, [lww[i] for i in ITEM_IDS], .4, label="LWW overwrite", color="tab:red")
plt.xticks(x, [f"{i + 1}" for i in range(12)]); plt.xlabel("item"); plt.ylabel("stock"); plt.legend(); plt.title("Merged stock: CRDT vs last-write-wins"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 04 예상 실행 결과 (Output)</b></summary>

```text
[CRDT 동기화] 노드 5곳 + 서버, 트랜잭션 2100건: 수렴 성공, 서버 재고와 정답(전체 합산) 차이 합계 0개 -> 갱신 유실 0건
병합 1회 평균 0.022ms, 최대 0.035ms, 전체 동기화 0.22ms (목표 200ms 이하: 달성)
숫자 덮어쓰기(LWW) 병합: 정답과 차이 합계 5142개 (갱신 유실) / PN-카운터 0개

[CRDT 성질 시험(무작위 상태 200쌍)] 멱등(두 번 병합) 200/200, 교환(A+B = B+A) 200/200, 결합((A+B)+C = A+(B+C)) 200/200, 오래된 상태 재전송 200/200

[충돌 탐지] 정답 충돌 226건 / 쌍 400건
  시간 근접 규칙(1시간 이내)               재현율  4.9%, 정밀도  9.3%, 오경보 107건, 놓침 215건 {'concurrent': 163, 'causal': 107, 'skew': 52}
  값 비교 규칙(다르면 충돌)                재현율 100.0%, 정밀도 67.9%, 오경보 107건, 놓침 0건 {'causal': 107}
  버전 벡터 규칙 v1(동시이면 충돌)           재현율 100.0%, 정밀도 88.3%, 오경보 30건, 놓침 0건 {'same_value': 30}
  버전 벡터 + 값 비교 v2(재설계)           재현율 100.0%, 정밀도 100.0%, 오경보 0건, 놓침 0건 {}
오경보·미탐 패턴: 시간 규칙은 오프라인 동안 멀리 떨어진 동시 수정(concurrent)·시계 어긋남(skew)을 놓치고, 값 비교 규칙은 '후속 정상 수정'(causal)을 충돌로 올림 -> 버전 벡터로 인과 관계를 보고 같은 값이면 제외(v2)
※ 충돌로 표시된 건은 자동 병합하지 않고 군수관이 수동 병합한다(M7). 재고 수량은 CRDT로 자동 합산되므로 사람이 판단할 대상은 메타 정보 충돌이다.
```
</details>
